# Resultados automáticos de extracción de claims

Este notebook construye las tablas comparativas de una corrida concreta de la primera fase del experimento, con el formato de una tabla de paper. Lee los `summary.csv` producidos por `scripts/evaluate_claims.py` para los cuatro modelos locales:

- T5S
- BARTO
- mBART-50
- uMT5-Base

La evaluación es automática y de referencia: METEOR, ROUGE-L F1 y BERTScore F1. El notebook también exporta un desglose complementario de precision, recall y F1 para ROUGE-L y BERTScore. No incluye revisión humana ni métricas de acuerdo entre anotadores.

## Interpretación y límites

Las métricas de referencia comparan cada predicción con `reference_claim`, que debe estar anotado antes de ejecutar la evaluación. No miden por sí solas la verdad médica del claim.

BERTScore F1 se calcula con `xlm-roberta-base` multilingüe, sin reescalado; aporta similitud semántica ante paráfrasis en español. Precision y recall de ROUGE-L/BERTScore describen, respectivamente, la precisión del contenido generado y la cobertura de la referencia; sus F1 se conservan en la tabla principal. Las tasas de salida vacía, claims de menos de cinco tokens, longitud media y duplicados son diagnósticos operativos; se reportan aparte y no se presentan como calidad factual.

Si todavía no existen resultados para la corrida indicada, el notebook muestra qué falta y no inventa valores.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display


def find_repository_root(start: Path | None = None) -> Path:
    """Encuentra la raíz desde el directorio de ejecución del notebook."""
    start = (start or Path.cwd()).resolve()
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / 'configs' / 'models.json').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError(
        'No se encontró la raíz del repositorio. Abre Jupyter desde el repositorio o desde notebooks/.'
    )


ROOT = find_repository_root()
# Debe coincidir con RUN_ID en 02_entrenar_modelos_sem1.ipynb.
RUN_ID = 'sem1_20261031_400_claims_v1'
RUN_DIR = ROOT / 'artifacts' / 'runs' / RUN_ID
EVALUATION_ROOT = RUN_DIR / 'evaluation'
TABLES_ROOT = EVALUATION_ROOT / 'tables'

print(f'Raíz del repositorio: {ROOT}')
print(f'Corrida: {RUN_ID}')
print(f'Directorio de evaluación: {EVALUATION_ROOT}')
print(f'Se guardarán las tablas en: {TABLES_ROOT}')

Raíz del repositorio: /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification
Corrida: sem1_20261031_400_claims_v1
Directorio de evaluación: /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1/evaluation
Se guardarán las tablas en: /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables


In [2]:
# Carga el registro para conservar el orden y los nombres oficiales de los modelos.
registry_payload = json.loads((ROOT / 'configs' / 'models.json').read_text(encoding='utf-8'))
registry = pd.DataFrame(
    [
        {
            'model_id': item['id'],
            'model_display_name': item['display_name'],
            'base_checkpoint': item['base_checkpoint'],
            'paper': item['paper'],
        }
        for item in registry_payload['models']
    ]
)


def discover_summary_files(root: Path) -> list[Path]:
    files: list[Path] = []
    direct = root / 'summary.csv'
    if direct.is_file():
        files.append(direct)
    files.extend(sorted(path for path in root.glob('*/summary.csv') if path.is_file()))
    return list(dict.fromkeys(files))


summary_files = discover_summary_files(EVALUATION_ROOT)
frames: list[pd.DataFrame] = []
for path in summary_files:
    frame = pd.read_csv(path)
    if frame.empty:
        continue
    if 'model_id' not in frame.columns:
        frame['model_id'] = path.parent.name
    frame['summary_source'] = str(path.relative_to(ROOT))
    frames.append(frame)

if frames:
    summaries = pd.concat(frames, ignore_index=True)
    # Si existe una salida directa y otra por modelo, conserva una sola fila por modelo.
    summaries = summaries.drop_duplicates(subset=['model_id'], keep='first')
else:
    summaries = pd.DataFrame(columns=['model_id'])

display_names = dict(zip(registry['model_id'], registry['model_display_name']))
if 'model_display_name' not in summaries.columns:
    summaries['model_display_name'] = summaries['model_id'].map(display_names)
else:
    summaries['model_display_name'] = summaries['model_display_name'].fillna(
        summaries['model_id'].map(display_names)
    )

order = {model_id: position for position, model_id in enumerate(registry['model_id'])}
if not summaries.empty:
    summaries['_order'] = summaries['model_id'].map(order).fillna(len(order))
    summaries = summaries.sort_values(['_order', 'model_id']).drop(columns='_order').reset_index(drop=True)

status = registry[['model_id', 'model_display_name', 'base_checkpoint', 'paper']].copy()
status['summary_found'] = status['model_id'].isin(set(summaries['model_id']))

display(Markdown('### Estado de los resultados'))
display(status)
if summaries.empty:
    display(Markdown(
        'Todavía no hay `summary.csv`. Primero entrena, infiere y evalúa los modelos; al volver a ejecutar este notebook se llenarán las tablas.'
    ))
else:
    display(Markdown(f'Se encontraron **{len(summaries)}** modelo(s) evaluado(s) en **{len(summary_files)}** archivo(s) de resumen.'))
    display(summaries)

### Estado de los resultados

,model_id,model_display_name,base_checkpoint,paper,summary_found
0,t5s_spanish,T5S,vgaraujov/t5-base-spanish,Spanish Pre-trained Language Models,True
1,barto_spanish,BARTO,vgaraujov/bart-base-spanish,Spanish Pre-trained Language Models,True
2,mbart50_es,mBART-50,facebook/mbart-large-50,AKCIT-FN at CheckThat! 2025,True
3,umt5_base_es,uMT5-Base,google/umt5-base,MMA at CheckThat! 2025,True


Se encontraron **4** modelo(s) evaluado(s) en **5** archivo(s) de resumen.

,model_id,model_display_name,prediction_n,reference_n,empty_rate,shorter_than_5_tokens_rate,mean_prediction_tokens_operational,duplicate_output_rate,meteor,rouge_l_precision,rouge_l_recall,rouge_l_f1,bertscore_precision,bertscore_recall,bertscore_f1,bertscore_model,mean_prediction_tokens_reference,summary_source
0,t5s_spanish,T5S,40,40,0.025,0.225,12.850,0.05,0.173504,0.355973,0.185653,0.220925,0.841211,0.809392,0.824626,xlm-roberta-base,12.850,artifacts/runs/sem1_20261031_400_claims_v1/eva...
1,barto_spanish,BARTO,40,40,0.000,0.000,18.850,0.00,0.315000,0.351639,0.316687,0.325780,0.903427,0.896149,0.899658,xlm-roberta-base,18.850,artifacts/runs/sem1_20261031_400_claims_v1/eva...
2,mbart50_es,mBART-50,40,40,0.000,0.000,19.325,0.00,0.325756,0.358986,0.332755,0.335128,0.904701,0.898072,0.901226,xlm-roberta-base,19.325,artifacts/runs/sem1_20261031_400_claims_v1/eva...
3,umt5_base_es,uMT5-Base,40,40,0.000,0.000,18.425,0.00,0.345111,0.397288,0.351832,0.363487,0.909671,0.901609,0.905471,xlm-roberta-base,18.425,artifacts/runs/sem1_20261031_400_claims_v1/eva...


## Tabla 1. Métricas automáticas de extracción

La negrita marca el mejor valor entre los modelos disponibles. Las flechas indican si un valor mayor (`↑`) o menor (`↓`) es preferible. `N ref.` es el número de predicciones que sí pudieron emparejarse con un `reference_claim`.

In [3]:
def highlight_extrema(data: pd.DataFrame, columns: list[str], maximize: bool) -> pd.DataFrame:
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for column in columns:
        if column not in data.columns:
            continue
        values = pd.to_numeric(data[column], errors='coerce')
        if values.notna().any():
            best = values.max() if maximize else values.min()
            styles.loc[values.eq(best), column] = 'font-weight: bold'
    return styles


metric_columns = ['meteor', 'rouge_l_f1', 'bertscore_f1']
for column in ['model_display_name', 'prediction_n', 'reference_n', *metric_columns]:
    if column not in summaries.columns:
        summaries[column] = pd.NA

table_automatic = summaries[
    ['model_display_name', 'prediction_n', 'reference_n', 'meteor', 'rouge_l_f1', 'bertscore_f1']
].copy()
table_automatic = table_automatic.rename(
    columns={
        'model_display_name': 'Modelo',
        'prediction_n': 'N pred.',
        'reference_n': 'N ref.',
        'meteor': 'METEOR (solapamiento léxico) ↑',
        'rouge_l_f1': 'ROUGE-L F1 (estructura) ↑',
        'bertscore_f1': 'BERTScore F1 (similitud semántica) ↑',
    }
)

table_automatic_style = (
    table_automatic.style
    .format(
        {
            'METEOR (solapamiento léxico) ↑': '{:.3f}',
            'ROUGE-L F1 (estructura) ↑': '{:.3f}',
            'BERTScore F1 (similitud semántica) ↑': '{:.3f}',
        },
        na_rep='—',
    )
    .apply(
        lambda data: highlight_extrema(
            data, [
                'METEOR (solapamiento léxico) ↑',
                'ROUGE-L F1 (estructura) ↑',
                'BERTScore F1 (similitud semántica) ↑',
            ], maximize=True
        ),
        axis=None,
    )
    .set_caption('Resultados automáticos contra claims de referencia')
)
display(table_automatic_style)
TABLES_ROOT.mkdir(parents=True, exist_ok=True)
table_automatic.to_csv(TABLES_ROOT / 'table_1_automatic_metrics.csv', index=False)
(TABLES_ROOT / 'table_1_automatic_metrics.html').write_text(
    table_automatic_style.to_html(), encoding='utf-8'
)
print(f'Tabla guardada en: {TABLES_ROOT / "table_1_automatic_metrics.csv"}')

,Modelo,N pred.,N ref.,METEOR (solapamiento léxico) ↑,ROUGE-L F1 (estructura) ↑,BERTScore F1 (similitud semántica) ↑
0,T5S,40,40,0.174,0.221,0.825
1,BARTO,40,40,0.315,0.326,0.900
2,mBART-50,40,40,0.326,0.335,0.901
3,uMT5-Base,40,40,0.345,0.363,0.905


Tabla guardada en: /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_1_automatic_metrics.csv


## Tabla 2. Diagnósticos operativos

Estos indicadores sirven para detectar fallos de generación o salidas degeneradas. No sustituyen METEOR, ROUGE-L ni BERTScore, y no deben interpretarse como fidelidad o veracidad.

In [4]:
diagnostic_columns = [
    'empty_rate',
    'shorter_than_5_tokens_rate',
    'mean_prediction_tokens_operational',
    'duplicate_output_rate',
]
for column in ['model_display_name', *diagnostic_columns]:
    if column not in summaries.columns:
        summaries[column] = pd.NA

table_diagnostics = summaries[
    ['model_display_name', *diagnostic_columns]
].copy().rename(
    columns={
        'model_display_name': 'Modelo',
        'empty_rate': 'Salidas vacías ↓',
        'shorter_than_5_tokens_rate': 'Menos de 5 tokens ↓',
        'mean_prediction_tokens_operational': 'Tokens medios',
        'duplicate_output_rate': 'Duplicados ↓',
    }
)

table_diagnostics_style = (
    table_diagnostics.style
    .format(
        {
            'Salidas vacías ↓': '{:.3f}',
            'Menos de 5 tokens ↓': '{:.3f}',
            'Tokens medios': '{:.2f}',
            'Duplicados ↓': '{:.3f}',
        },
        na_rep='—',
    )
    .apply(
        lambda data: highlight_extrema(
            data,
            ['Salidas vacías ↓', 'Menos de 5 tokens ↓', 'Duplicados ↓'],
            maximize=False,
        ),
        axis=None,
    )
    .set_caption('Diagnósticos operativos de la generación')
)
display(table_diagnostics_style)
table_diagnostics.to_csv(TABLES_ROOT / 'table_2_operational_diagnostics.csv', index=False)
(TABLES_ROOT / 'table_2_operational_diagnostics.html').write_text(
    table_diagnostics_style.to_html(), encoding='utf-8'
)
print(f'Tabla guardada en: {TABLES_ROOT / "table_2_operational_diagnostics.csv"}')

,Modelo,Salidas vacías ↓,Menos de 5 tokens ↓,Tokens medios,Duplicados ↓
0,T5S,0.025,0.225,12.85,0.050
1,BARTO,0.000,0.000,18.85,0.000
2,mBART-50,0.000,0.000,19.32,0.000
3,uMT5-Base,0.000,0.000,18.43,0.000


Tabla guardada en: /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_2_operational_diagnostics.csv


## Tabla 3. Desglose de precision, recall y F1

Esta tabla es complementaria: ROUGE-L mide coincidencia estructural basada en subsecuencias y BERTScore similitud semántica contextual. No equivale a precision, recall y F1 de una clasificación binaria.

In [5]:
prf_columns = [
    'rouge_l_precision', 'rouge_l_recall', 'rouge_l_f1',
    'bertscore_precision', 'bertscore_recall', 'bertscore_f1',
]
for column in ['model_display_name', *prf_columns]:
    if column not in summaries.columns:
        summaries[column] = pd.NA

table_prf = summaries[['model_display_name', *prf_columns]].copy().rename(
    columns={
        'model_display_name': 'Modelo',
        'rouge_l_precision': 'ROUGE-L Precision (contenido generado) ↑',
        'rouge_l_recall': 'ROUGE-L Recall (cobertura de referencia) ↑',
        'rouge_l_f1': 'ROUGE-L F1 ↑',
        'bertscore_precision': 'BERTScore Precision (similitud generada) ↑',
        'bertscore_recall': 'BERTScore Recall (cobertura semántica) ↑',
        'bertscore_f1': 'BERTScore F1 ↑',
    }
)

table_prf_style = (
    table_prf.style
    .format({column: '{:.3f}' for column in table_prf.columns if column != 'Modelo'}, na_rep='—')
    .apply(
        lambda data: highlight_extrema(
            data, [column for column in table_prf.columns if column != 'Modelo'], maximize=True
        ),
        axis=None,
    )
    .set_caption('Desglose complementario de precision, recall y F1')
)
display(table_prf_style)
table_prf.to_csv(TABLES_ROOT / 'table_3_precision_recall_f1.csv', index=False)
(TABLES_ROOT / 'table_3_precision_recall_f1.html').write_text(
    table_prf_style.to_html(), encoding='utf-8'
)
print(f"Tabla guardada en: {TABLES_ROOT / 'table_3_precision_recall_f1.csv'}")

,Modelo,ROUGE-L Precision (contenido generado) ↑,ROUGE-L Recall (cobertura de referencia) ↑,ROUGE-L F1 ↑,BERTScore Precision (similitud generada) ↑,BERTScore Recall (cobertura semántica) ↑,BERTScore F1 ↑
0,T5S,0.356,0.186,0.221,0.841,0.809,0.825
1,BARTO,0.352,0.317,0.326,0.903,0.896,0.900
2,mBART-50,0.359,0.333,0.335,0.905,0.898,0.901
3,uMT5-Base,0.397,0.352,0.363,0.910,0.902,0.905


Tabla guardada en: /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_3_precision_recall_f1.csv


## Lectura final

Para el informe, usa la Tabla 1 como comparación principal. La Tabla 2 documenta problemas de ejecución y repetición; la Tabla 3 desglosa precision, recall y F1 de las métricas de texto. No se incluye una matriz de confusión porque la tarea es generación de texto, no clasificación binaria.

Este notebook tampoco afirma que un claim sea verdadero o falso: solo compara la salida generada con el claim de referencia y documenta el comportamiento operativo de cada modelo.

In [6]:
expected_models = set(registry['model_id'])
available_models = set(summaries['model_id'])
missing_models = [model_id for model_id in registry['model_id'] if model_id not in available_models]

if missing_models:
    display(Markdown(
        '**Pendientes:** todavía no hay resultados para: '
        + ', '.join(f'`{model_id}`' for model_id in missing_models)
        + '. Ejecuta su entrenamiento, inferencia y evaluación, y vuelve a correr el notebook.'
    ))
else:
    display(Markdown('**Listo:** los cuatro modelos tienen un resumen de evaluación disponible.'))

print('Archivos generados:')
for path in sorted(TABLES_ROOT.glob('table_*')):
    print(f'- {path.relative_to(ROOT)}')

**Listo:** los cuatro modelos tienen un resumen de evaluación disponible.

Archivos generados:
- artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_1_automatic_metrics.csv
- artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_1_automatic_metrics.html
- artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_2_operational_diagnostics.csv
- artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_2_operational_diagnostics.html
- artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_3_precision_recall_f1.csv
- artifacts/runs/sem1_20261031_400_claims_v1/evaluation/tables/table_3_precision_recall_f1.html
